ARTI308 - Machine Learning

# Lab 4: Data Quality Assessment & Preprocessing

In real-world machine learning projects, data is often:
- Incomplete (missing values)
- Noisy (outliers or random errors)
- Inconsistent (wrong formats, mixed units)

Before building any machine learning model, we must clean and prepare the data properly.

In this lab, we will apply practical preprocessing techniques step by step using the **AI Job Market Dataset**.

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='whitegrid')

## 1. Load Dataset

In [ ]:
df = pd.read_csv('AIJobMarket.csv')

In [ ]:
df.head()

## Task 1: Identify Data Quality Issues

### 1.1 Check Shape and Basic Info

In [ ]:
print('Dataset Shape:', df.shape)
print()
df.info()

### 1.2 Check Data Types

Data types must match the real meaning of each column.
For example:
- `job_title`, `company_size`, `country` should be categorical (object)
- `salary`, `years_experience`, `job_openings` should be numeric

In [ ]:
df.dtypes

**Observation:** 
- All numerical columns (`salary`, `years_experience`, `job_openings`) are correctly stored as `int64`.
- Categorical columns (`job_title`, `company_size`, `country`, etc.) are stored as `object`, which is acceptable.
- No major data type issues were found. The dataset is well-structured.

However, we can convert categorical columns to the `category` dtype for better memory efficiency:

In [ ]:
# Convert relevant columns to category dtype
cat_cols = ['job_title', 'company_size', 'company_industry', 'country',
            'remote_type', 'experience_level', 'education_level', 'hiring_urgency']

for col in cat_cols:
    df[col] = df[col].astype('category')

df.dtypes

**Result:** Categorical columns are now stored as `category` dtype, which is more memory-efficient and semantically correct.

### 1.3 Check for Duplicate Rows

In [ ]:
print('Number of duplicate rows:', df.duplicated().sum())

### 1.4 Check Value Ranges

In [ ]:
df[['years_experience', 'salary', 'job_openings']].describe()

**Summary of Data Quality Issues Found:**

1. Categorical columns were stored as `object` instead of `category` → **Fixed above**
2. The `salary` column has a wide range (min: ~45K, max: ~204K) → potential **outliers** to investigate in Task 3
3. No missing values were found in the original dataset → we will **introduce artificial missing values** in Task 2 for learning purposes

---
## Task 2: Handling Missing Values

### 2.1 Detect Missing Values

In [ ]:
df.isna().sum()

The dataset has no missing values. We will introduce artificial missing values in the `salary` column for demonstration purposes.

### 2.2 Introduce Artificial Missing Values

In [ ]:
df2 = df.copy()
df2.loc[0:5, 'salary'] = np.nan

print('Missing values after introduction:')
print(df2.isna().sum())

In [ ]:
df2.head(10)

### 2.3 Apply Missing Value Strategy: Median Imputation

**Chosen Strategy: Median Imputation**

**Why Median over Mean?**

The `salary` column has a wide range (min: ~45K, max: ~204K) with high-earning outlier roles like senior AI engineers. The median is more robust to these extreme values than the mean.

If we used mean imputation, high salaries would pull the average upward and overestimate the "typical" salary. The median gives us a more realistic central value to fill in the missing entries.

In [ ]:
df_median = df2.copy()
median_salary = df_median['salary'].median()
print(f'Median salary used for imputation: ${median_salary:,.2f}')

df_median['salary'].fillna(median_salary, inplace=True)

print()
print('Missing values after imputation:')
print(df_median.isna().sum())

In [ ]:
df_median.head(10)

**Result:** The 6 missing salary values have been replaced with the median salary. The dataset size is preserved and no rows were lost.

---
## Task 3: Detect and Handle Outliers Using IQR

We will detect outliers in the `salary` column using the Interquartile Range (IQR) method.

The IQR method defines outliers as values outside:
- `Q1 - 1.5 × IQR` (lower bound)
- `Q3 + 1.5 × IQR` (upper bound)

In [ ]:
# Visualize salary distribution with boxplot
plt.figure(figsize=(8, 4))
sns.boxplot(x=df['salary'])
plt.title('Boxplot of Salary')
plt.xlabel('Salary (USD)')
plt.show()

### 3.1 Detect Outliers

In [ ]:
Q1 = df['salary'].quantile(0.25)
Q3 = df['salary'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f'Q1: ${Q1:,.2f}')
print(f'Q3: ${Q3:,.2f}')
print(f'IQR: ${IQR:,.2f}')
print(f'Lower Bound: ${lower:,.2f}')
print(f'Upper Bound: ${upper:,.2f}')

outliers = df[(df['salary'] < lower) | (df['salary'] > upper)]
print(f'\nNumber of outliers detected: {len(outliers)}')
outliers[['job_title', 'experience_level', 'salary']].head(15)

### 3.2 Handle Outliers: Capping (Percentile Method)

**Chosen Strategy: Capping at 5th and 95th Percentile**

**Why Capping instead of Removal?**

In a job market dataset, extreme salaries are **valid real-world data** — a very high salary may represent a senior AI director, and a very low salary may represent an intern or part-time role. Removing these records would cause us to lose important market information.

Instead, capping brings extreme values to the boundary of the acceptable range while preserving all records.

In [ ]:
lower_cap = df['salary'].quantile(0.05)
upper_cap = df['salary'].quantile(0.95)

print(f'Lower cap (5th percentile): ${lower_cap:,.2f}')
print(f'Upper cap (95th percentile): ${upper_cap:,.2f}')

df_capped = df.copy()
df_capped['salary'] = df_capped['salary'].clip(lower_cap, upper_cap)

In [ ]:
# Compare before and after
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(x=df['salary'], ax=axes[0])
axes[0].set_title('Before Capping')
axes[0].set_xlabel('Salary (USD)')

sns.boxplot(x=df_capped['salary'], ax=axes[1])
axes[1].set_title('After Capping (5th-95th Percentile)')
axes[1].set_xlabel('Salary (USD)')

plt.tight_layout()
plt.show()

**Result:** Extreme salary values have been capped at the 5th and 95th percentiles. The dataset size remains the same (no rows removed), and the salary distribution is now less influenced by extreme values.

---
## Task 4: Data Transformation – Normalization

We will normalize `salary` and `years_experience` using both Min-Max and Z-Score methods.

### 4.1 Min-Max Normalization

Min-Max normalization rescales values to the range [0, 1] using the formula:

$$X_{scaled} = \frac{X - X_{min}}{X_{max} - X_{min}}$$

In [ ]:
df_capped[['salary', 'years_experience']].head()

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df_minmax = df_capped[['salary', 'years_experience']].copy()
df_minmax[['salary', 'years_experience']] = scaler.fit_transform(df_minmax)

print('Min-Max Normalized:')
df_minmax.head()

In [ ]:
print('Min-Max range check:')
print(df_minmax.describe().loc[['min','max']])

**Result:** All values are now in the range [0, 1]. A salary of 0 represents the minimum observed salary and 1 represents the maximum. This is useful for distance-based models like KNN or K-Means.

### 4.2 Z-Score Standardization

Z-score standardization transforms data so that the mean becomes 0 and the standard deviation becomes 1:

$$Z = \frac{X - \mu}{\sigma}$$

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_standardized = df_capped[['salary', 'years_experience']].copy()
df_standardized[['salary', 'years_experience']] = scaler.fit_transform(df_standardized)

print('Z-Score Standardized:')
df_standardized.head()

In [ ]:
print('Z-Score stats check (mean ≈ 0, std ≈ 1):')
print(df_standardized.describe().loc[['mean','std']])

**Result:** After standardization, both features are centered around 0 with a standard deviation of 1. Values above the mean are positive and below the mean are negative. This is suitable for Linear Regression, SVM, and PCA.

---
## Task 5: Check Correlation Before Applying PCA

We first check whether `salary` and `years_experience` are correlated.
PCA is only meaningful if features show a strong correlation (overlapping information).

In [ ]:
plt.figure(figsize=(6, 4))
sns.heatmap(
    df_standardized[['salary', 'years_experience']].corr(),
    annot=True,
    cmap='coolwarm',
    fmt='.3f'
)
plt.title('Correlation Heatmap (Before PCA)')
plt.show()

**Observation:** The correlation between `salary` and `years_experience` is approximately **-0.013**, which is very close to 0.

This means there is almost **no linear relationship** between salary and years of experience in this dataset — a surprising but realistic finding, as AI job salaries are driven more by role type, company, and skills rather than years alone.

Since the correlation is near zero, PCA would not provide meaningful dimensionality reduction here. However, we apply it below **for demonstration purposes**.

### Apply PCA (Demonstration)

In [ ]:
from sklearn.decomposition import PCA

X = df_standardized[['salary', 'years_experience']]

pca = PCA(n_components=2)
principal_components = pca.fit_transform(X)

print('Explained Variance Ratio:', pca.explained_variance_ratio_)
print(f'PC1 explains: {pca.explained_variance_ratio_[0]*100:.1f}%')
print(f'PC2 explains: {pca.explained_variance_ratio_[1]*100:.1f}%')

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(principal_components[:, 0], principal_components[:, 1],
            alpha=0.3, s=10, color='steelblue')
plt.title('PCA Projection — AI Job Market Dataset')
plt.xlabel(f'Principal Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'Principal Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.show()

**Conclusion on PCA:**

The explained variance is split roughly equally between PC1 and PC2 (~50% each). This confirms that neither component dominates, meaning the two features (`salary` and `years_experience`) carry **independent information** and are not redundant.

In this case, **PCA does not offer a benefit** — keeping both original features separately is more informative than reducing to one component. This aligns with the near-zero correlation we observed in the heatmap.

**PCA would be more useful** if we included strongly correlated skill columns (e.g., `skills_python`, `skills_ml`, `skills_deep_learning`) which likely overlap in the types of candidates they describe.

---
# End of Lab 4 Assignment